# Graded evaluation: none / icl / arm_c

Deterministic `silent_break` on the official seeds, paired with real
`no_change` instances so detection is a sensitivity and specificity pair.
Everything is scored by `ecpm_parser` at the environment repo, so these
numbers are comparable with Efe's pilot table and with the agentic arm.

Replaces `armc_official_eval.ipynb`, whose route column is void: it
hard-coded `START, GOAL = "E", "F"`, and only seed 7 has that start and
goal. Across the 32 official seeds there are 19 distinct pairs, so the
route walk began at the wrong node on 31 of them.

## Notes on this run

**One model, adapter toggled.** The base is loaded once and wrapped with
the adapter; `none` and `icl` run inside `model.disable_adapter()`. That
halves memory against loading two copies and removes any doubt that
`icl` and `arm_c` differ only by the training.

**Resumable.** Rows stream to JSONL in `/kaggle/working` as they are
produced. If the session dies, rerun the cell and it picks up where it
stopped. The full run is roughly 650 generations, so budget two hours;
the timing probe below gives a real estimate before you commit.

**Preservation reports three numbers.** Accuracy, the constant-answer
baseline on the same instances, and target-pair recall. Accuracy alone is
met by answering "nothing changed" everywhere, which is what the August
adapter did on all 64 replies.

**`bare_json` is a diagnostic, not a score.** The frozen contract takes
the first balanced parseable `{...}` and ignores prose around it. This
column records whether the model needed that leniency.

In [ ]:
import glob
for m in ("resource_mdp.py", "anchors_v22.py", "baselines.py",
          "adapter_config.json", "payloads_silent_break_det",
          "payloads_no_change_det"):
    hits = glob.glob(f"/kaggle/input/**/{m}", recursive=True)
    print(f"{m:28} {hits[:1] or 'MISSING'}")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    """Directory containing `marker`, at any depth. Kaggle nests datasets
    under /kaggle/input/datasets/<user>/<slug>/ in some workspaces."""
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

REPO_PATH    = find_dir("resource_mdp.py")
EVAL_PATH    = find_dir("anchors_v22.py")
ADAPTER_PATH = find_dir("adapter_config.json")
PAY_CHANGED  = sorted(glob.glob(
    "/kaggle/input/**/payloads_silent_break_det", recursive=True))[0]
PAY_NOCHANGE = sorted(glob.glob(
    "/kaggle/input/**/payloads_no_change_det", recursive=True))[0]
OUT_DIR = "/kaggle/working"

print("repo:    ", REPO_PATH)
print("eval:    ", EVAL_PATH)
print("adapter: ", ADAPTER_PATH)
print("changed: ", PAY_CHANGED)
print("nochange:", PAY_NOCHANGE)

for f in ("run_pilot.py", "ecpm_parser.py", "explore_agent.py",
          "explore_metrics.py"):
    assert os.path.isfile(os.path.join(REPO_PATH, f)), f"{f} missing from repo"

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
# is_bf16_supported() counts emulation and returns True on a T4 (7.5),
# which has no native bfloat16. Go by compute capability.
USE_BF16 = CAP[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"\ngpu: {GPU} | capability {CAP[0]}.{CAP[1]} | compute dtype: {DTYPE}")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

## Adapter identity

The adapter shipped in the August handoff was a Qwen2.5-**3B** LoRA at
`lora_dropout` 0.05. Loading a 3B adapter on a 1.5B base either errors or
mismatches silently, so check before anything else.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

cfg = json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json")))
assert cfg["base_model_name_or_path"] == MODEL_NAME, (
    f"adapter trained on {cfg['base_model_name_or_path']}, "
    f"this notebook loads {MODEL_NAME}")
assert cfg["lora_dropout"] == 0.0, cfg["lora_dropout"]
print("adapter ok:", cfg["base_model_name_or_path"],
      "| r", cfg["r"], "| alpha", cfg["lora_alpha"])

prov_path = os.path.join(ADAPTER_PATH, "phase1_provenance.json")
if os.path.isfile(prov_path):
    prov = json.load(open(prov_path))
    print("phase 1:", prov["anchor_worlds"], "worlds,",
          prov["anchor_examples"], "examples,",
          prov["optimizer_steps"], "steps, loss",
          round(prov["final_loss"], 4))
else:
    print("no phase1_provenance.json next to the adapter")

## Payloads

`common_seeds` matters. `no_change` constructs on all 80 seeds and the
break family on 65, so an unmatched pairing would compare different
graphs across the two detection conditions.

In [ ]:
N_SEEDS = 32          # set to 8 for a quick first pass, then raise

sys.path.insert(0, EVAL_PATH)
import ecpm_eval as E
E.attach(REPO_PATH)

changed  = E.load_payloads(PAY_CHANGED)
nochange = E.load_payloads(PAY_NOCHANGE)
seeds = E.common_seeds(changed, nochange)[:N_SEEDS]
changed  = [p for p in changed  if p["seed"] in seeds]
nochange = [p for p in nochange if p["seed"] in seeds]
payloads = changed + nochange

n_probes = sum(len(p["probes"]) for p in payloads)
print(f"{len(seeds)} matched seeds: {seeds}")
print(f"{len(payloads)} payloads, {n_probes} probes per arm, "
      f"{n_probes * 3} generations total")
print("distinct start/goal in this set:",
      len({(p['facts']['start'], p['facts']['goal']) for p in changed}))

## The ceiling, before any model number

A model at or below `rule_drop` has not beaten counting. On deterministic
`silent_break` the rule is 1.00, so anything here is a capability
threshold rather than a world model. Worth having on screen next to the
model's numbers.

In [ ]:
import subprocess
print(subprocess.run([sys.executable,
                      os.path.join(EVAL_PATH, "baselines.py"), PAY_CHANGED],
                     capture_output=True, text=True).stdout)

## Model

Loaded once. `arm_c` runs with the adapter active, `icl` and `none` run
inside `disable_adapter()`, so the only difference between `icl` and
`arm_c` is phase 1.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
model.config.use_cache = True
print("loaded | GPU allocated GB:",
      round(torch.cuda.memory_allocated() / 1e9, 2))

def generate(messages, max_new_tokens):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)

def make_ask(use_adapter):
    def ask(messages, max_new_tokens):
        if use_adapter:
            return generate(messages, max_new_tokens)
        with model.disable_adapter():
            return generate(messages, max_new_tokens)
    return ask

ASK = {"none": make_ask(False), "icl": make_ask(False),
       "arm_c": make_ask(True)}

# sanity: the adapter really is doing something
probe_msgs = [{"role": "system", "content": E.SYSTEM},
              {"role": "user", "content": changed[0]["single"]["detection"]}]
print("base :", repr(ASK["icl"](probe_msgs, 32)))
print("armc :", repr(ASK["arm_c"](probe_msgs, 32)))

## Timing probe

One payload through one arm, timed, so you know whether the full run fits
the session before starting it.

In [ ]:
t0 = time.time()
_ = E.run_arm([changed[0]], ASK["arm_c"], arm="_timing", mode="single",
              verbose=False)
per_payload = time.time() - t0
total_min = per_payload * len(payloads) * 3 / 60
print(f"{per_payload:.1f}s per payload per arm "
      f"-> about {total_min:.0f} minutes for all three arms")
if total_min > 300:
    print("\nThat will not fit one session. Lower N_SEEDS to 16 and rerun "
          "from the payload cell, or run one arm at a time; the JSONL "
          "resumes either way.")

## Run

`arm_c` first, then `icl`, then `none`, so a timeout still leaves the two
arms that carry the comparison. Rerunning the cell resumes from the
JSONL.

In [ ]:
ARMS = ["arm_c", "icl", "none"]

rows = []
for arm in ARMS:
    print(f"\n=== {arm} ===")
    t0 = time.time()
    rows += E.run_arm(
        payloads, ASK[arm], arm=arm, mode="single",
        with_evidence=(arm != "none"),
        out_path=os.path.join(OUT_DIR, f"raw_single_{arm}.jsonl"))
    print(f"  {arm} done in {(time.time()-t0)/60:.1f} min")
print(f"\n{len(rows)} rows")

## Results

`sens` and `spec` are the detection pair. A constant answerer scores 1.00
on one and 0.00 on the other, which is why a single detection number was
uninterpretable.

`pres_acc` next to `pres_const` is the point: if they match, the arm
answered the majority class. `target_recall` is the informative part
underneath.

In [ ]:
import pandas as pd

COLS = ["arm", "sens", "spec", "localize", "pres_acc", "pres_const",
        "target_recall", "pres_parsed", "route_valid", "route_optimal",
        "mean_regret", "bare_json"]
tab = E.table(rows, arms=ARMS)
display(pd.DataFrame(tab).reindex(columns=COLS))

print("\nroute status by arm")
for r in tab:
    print(" ", r["arm"], r["route_status"])

print("\nMcNemar on localization, exact two-sided")
for a, b in (("arm_c", "icl"), ("icl", "none")):
    print(f"  {a} vs {b}: "
          f"{E.mcnemar([r for r in rows if r['arm']==a], [r for r in rows if r['arm']==b])}")

json.dump(tab, open(os.path.join(OUT_DIR, "table_single.json"), "w"), indent=1)

## Per-seed localization

This is how the old 31/32 was cross-checked against independently
recorded break pairs. `node_only` counts right-node-wrong-action, which
was the shape of the single miss last time.

In [ ]:
loc = {}
for r in rows:
    if r["probe"] == "localization" and r["scored"].get("applicable"):
        p = r["parsed"]
        said = (f"{p.get('node')} {p.get('action')}"
                if p["status"] == "ok" else p["status"])
        loc.setdefault(r["seed"], {})[r["arm"]] = said
        loc[r["seed"]]["gold"] = " ".join(r["target"])
df = pd.DataFrame(loc).T[["gold"] + ARMS]
display(df)

for arm in ARMS:
    exact = sum(df[arm] == df["gold"])
    node = sum(a.split()[0] == g.split()[0]
               for a, g in zip(df[arm], df["gold"]) if " " in a)
    print(f"  {arm:6} exact {exact}/{len(df)}   node-level {node}/{len(df)}")

## Package the outputs

In [ ]:
import zipfile, shutil

ZIP_PATH = f"{OUT_DIR}/eval_all.zip"
SKIP = ("eval_all.zip", ".ipynb_checkpoints")
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if not os.path.isfile(path):
            continue
        rel = os.path.relpath(path, OUT_DIR)
        if any(s in rel for s in SKIP):
            continue
        z.write(path, rel)
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")
for i in zipfile.ZipFile(ZIP_PATH).namelist():
    print("  ", i)